# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is hosted at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL for the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, their `@id`s, and the contained fields and columns.

We will list all record sets in the dataset, and for each record set, show its fields and columns.

In [ ]:
# List all record sets and the available fields/columns with their @id
record_sets = [r for r in metadata.recordSet]
print(f"Found {len(record_sets)} record set(s):")
for idx, record_set in enumerate(record_sets):
    print(f"\nRecord Set #{idx+1}: {record_set.name}")
    print(f"  @id: {record_set['@id']}\n  Description: {getattr(record_set, 'description', 'N/A')}")
    # List out fields
    if hasattr(record_set, 'field') and record_set.field:
        print("  Fields:")
        for field in record_set.field:
            print(f"    - {field['@id']}: {field.name} (dataType: {getattr(field, 'dataType', 'N/A')})")
    # List out columns
    if hasattr(record_set, 'column') and record_set.column:
        print("  Columns:")
        for column in record_set.column:
            print(f"    - {column['@id']}: {column.name} (dataType: {getattr(column, 'dataType', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> **Note:** Use `@id` fields for referencing record sets and columns as recommended by the Croissant specification.

In [ ]:
# Record set selection (by @id):
record_set_ids = [rs['@id'] for rs in metadata.recordSet]
# For this dataset, there's likely one main record set. For demonstration, select the first:
main_record_set_id = record_set_ids[0]

# Extract records into a pandas DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]

print(f"Loaded DataFrame for record set {main_record_set_id} with shape {df.shape}")
print("Columns (@id):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA on the main record set.

We'll:
- Select a numeric field (column) for filtering and normalization.
- Filter records with high values in a chosen numeric field.
- Normalize the field.
- Group data by another field if relevant.

Scroll up to the "Data Overview" output to pick column `@id`s.


In [ ]:
# Suppose our numeric field of interest is peptide count (column '@id': 'peptide_count'),
# and we want to group by 'modification' if present.
# Update these with actual @id values from overview above as needed.

numeric_field_id = None
group_field_id = None

# Try to automatically pick a likely numeric and group field
for c in df.columns:
    if c.lower().startswith('peptide_count') or c.lower().endswith('count'):
        numeric_field_id = c
    if c.lower().startswith('modification') or c.lower().endswith('modification'):
        group_field_id = c

# Fallback to the first numeric field if not found
if numeric_field_id is None:
    sample_numeric = df.select_dtypes(include='number').columns
    numeric_field_id = sample_numeric[0] if len(sample_numeric) > 0 else df.columns[0]

print(f"Selected numeric field @id: {numeric_field_id}")
if group_field_id:
    print(f"Selected group field @id: {group_field_id}")

# Filter by threshold
threshold = 10
filtered_df = df.copy()
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
else:
    print(f"Warning: {numeric_field_id} is not detected as numeric. Displaying original data.")

print(filtered_df.head())

# Normalize
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id and group_field_id in filtered_df:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of our selected numeric field after filtering, and if grouped, show the means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set(style='whitegrid')

# Histogram of numeric field after filtering
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id exists, show bar plot of means
if group_field_id and group_field_id in filtered_df:
    plt.figure(figsize=(10,4))
    order = grouped_df.sort_values(numeric_field_id, ascending=False)[group_field_id].head(15)
    sns.barplot(data=grouped_df[grouped_df[group_field_id].isin(order)],
                x=group_field_id, y=numeric_field_id, order=order)
    plt.xticks(rotation=90)
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 15)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook has demonstrated how to use `mlcroissant` to load, explore, and analyze the FAIR² mass spectrometry dataset.

- We loaded metadata and reviewed available record sets and fields using their `@id`s following Croissant best practice.
- We extracted data into pandas DataFrames for convenient processing and explored field distributions.
- Basic filtering, normalization, grouping, and visualization was performed.

You can build upon this notebook to conduct further protein-level analysis, visualizations, and machine learning tasks using the curated and standardized dataset structure provided by Croissant.

---